In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Milestone 4


In [2]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
 
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv') 
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
OPTION_COLS = ["A", "B", "C", "D", "E"]
 
print(train.shape, test.shape)

(2000, 8) (500, 7)


In [3]:
train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


Q1. Label Encoding
Convert the answer column in train.csv into numeric labels using the following mapping:
A = 0
B = 1
C = 2
D = 3
E = 4

What is the encoded numeric label for the row at index 150?

In [4]:
label_map = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
train["label"] = train["answer"].map(label_map)
 
print("Encoded label at index 150:", train["label"].iloc[150])

Encoded label at index 150: 2


Q2. Prompt-Option Formatting
For row index 0, create the Option B input using exactly this format:
str(prompt) + " [SEP] " + str(option_B)

What is the exact character length of this formatted input string?

In [5]:
def format_option(row, option_col):
    return str(row["prompt"]) + " [SEP] " + str(row[option_col])
 
row0 = train.iloc[0]
formatted_B = format_option(row0, "B")
print("Formatted length for row 0, option B:", len(formatted_B))

Formatted length for row 0, option B: 407


Q3. Single-Row MCQ Tokenization
Using bert-base-uncased, tokenize the five formatted inputs for row index 0 with:
padding = "max_length"
truncation = True
max_length = 128
return_tensors = "pt"

After reshaping for a multiple-choice model, the final input_ids tensor has shape:
[1, 5, 128]

What is the value of the second dimension?

In [6]:
MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LEN = 128
 
def tokenize_row(row, tokenizer, max_length=128):
    prompts = [str(row["prompt"])] * 5         
    options = [str(row[c]) for c in OPTION_COLS]  
    enc = tokenizer(
        prompts,
        options,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )
   
    return {k: v.unsqueeze(0) for k, v in enc.items()}
 
enc0 = tokenize_row(row0, tokenizer, MAX_LEN)
print("input_ids shape:", enc0["input_ids"].shape)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

input_ids shape: torch.Size([1, 5, 128])


Q4. Batch MCQ Tokenization
Tokenize the first 16 rows of train.csv as multiple-choice examples.
Each row has 5 choices.
Each choice is tokenized to length 128.

The final input_ids tensor has shape:
[16, 5, 128]

How many total token positions are in this tensor?

In [7]:
def tokenize_batch(df, tokenizer, n_rows, max_length=128):
    df_batch = df.head(n_rows)
    
    prompts = [str(row["prompt"]) for _ in range(5) for _, row in df_batch.iterrows()]
    options = [str(row[c]) for _, row in df_batch.iterrows() for c in OPTION_COLS]
    
   
    enc = tokenizer(
        prompts, 
        options,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )
   
    return {k: v.view(n_rows, 5, max_length) for k, v in enc.items()}



batch16 = tokenize_batch(train, tokenizer, 16, MAX_LEN)
shape = batch16["input_ids"].shape
total_positions = shape[0] * shape[1] * shape[2]

print("Shape:", shape, "Total token positions:", total_positions)

Shape: torch.Size([16, 5, 128]) Total token positions: 10240


Q5. Multiple-Choice Logits
Load bert-base-uncased using AutoModelForMultipleChoice.
Tokenize row index 0 as 5 choices and pass it through the model.

The output logits tensor has shape:
[1, 5]

How many logits are produced for one question?

In [8]:
model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)
model.eval()

# Forward pass
with torch.no_grad():
    outputs = model(**enc0)

print("Logits shape:", outputs.logits.shape)      # Expected: [1, 5]
print("Number of choices:", outputs.logits.shape[1])

# Compute loss
labels = torch.tensor([train["label"].iloc[0]])

with torch.no_grad():
    outputs_with_loss = model(**enc0, labels=labels)

print("Loss:", outputs_with_loss.loss.item())


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Logits shape: torch.Size([1, 5])
Number of choices: 5
Loss: 1.611057996749878


Q6. For row index 0, pass the tokenized 5-choice input into AutoModelForMultipleChoice along with the correct encoded label.

The model returns a scalar loss tensor.

How many dimensions does this loss tensor have?
 

In [9]:
print("ndim:", outputs_with_loss.loss.dim())

ndim: 0


Q7. LoRA Trainable Parameters
Apply LoRA to the bert-base-uncased multiple-choice model using:
r = 8
lora_alpha = 16
target_modules = ["query", "value"]
lora_dropout = 0.1
bias = "none"
task_type = TaskType.SEQ_CLS

Count trainable parameters using:
sum(p.numel() for p in model.parameters() if p.requires_grad)

How many parameters are trainable?

In [10]:
base_model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

lora_model = get_peft_model(base_model, lora_config)

# Print summary
lora_model.print_trainable_parameters()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 295,681 || all params: 109,778,690 || trainable%: 0.2693


Q8. Create a Hugging Face Dataset from the first 100 rows of train.csv.

For each row, create:
input_ids with shape [5, 128]
attention_mask with shape [5, 128]
labels as the encoded answer label

For the first dataset item, input_ids has shape:
[5, 128]

How many tokenized choices are stored in input_ids?

In [11]:
from datasets import Dataset

def build_hf_dataset(df, tokenizer, max_length=128):
    def process_row(row):
        prompts = [str(row["prompt"])] * 5
        options = [str(row[c]) for c in OPTION_COLS]
        
        enc = tokenizer(prompts, options, 
                       padding="max_length", 
                       truncation=True, 
                       max_length=max_length)
        
        return {
            "input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "labels": label_map[row["answer"]]
        }
    
    records = [process_row(row) for _, row in df.iterrows()]
    return Dataset.from_list(records)

In [12]:
hf_dataset = build_hf_dataset(train.head(100), tokenizer, MAX_LEN)
first = hf_dataset[0]

print("input_ids shape:", len(first["input_ids"]), "x", len(first["input_ids"][0]))

input_ids shape: 5 x 128


Q9. Tiny LoRA Fine-Tuning
Fine-tune a LoRA multiple-choice model on the first 32 rows using Hugging Face Trainer.

Use the following settings:
max_length = 64
per_device_train_batch_size = 4
gradient_accumulation_steps = 1
max_steps = 4

What is the final global_step reported by the Trainer?

In [13]:
small_dataset = build_hf_dataset(
    train.head(32),
    tokenizer,
    max_length=64
)

# Custom Data Collator
# Combines individual samples into a batch of shape:
# input_ids      -> [batch_size, 5, 64]
# attention_mask -> [batch_size, 5, 64]
# labels         -> [batch_size]

from dataclasses import dataclass
import torch

@dataclass
class MCQCollator:
    def __call__(self, features):
        return {
            "input_ids": torch.tensor(
                [f["input_ids"] for f in features],
                dtype=torch.long
            ),
            "attention_mask": torch.tensor(
                [f["attention_mask"] for f in features],
                dtype=torch.long
            ),
            "labels": torch.tensor(
                [f["labels"] for f in features],
                dtype=torch.long
            )
        }

# Training Configuration

training_args = TrainingArguments(
    output_dir="./mcq_lora_tiny",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    report_to="none"
)

# Load BERT and Apply LoRA


base_model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)
lora_model = get_peft_model(base_model, lora_config)

# Create Trainer

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=small_dataset,
    data_collator=MCQCollator(),
)

# Train the Model


trainer.train()

# Final Training Step

print("Final global_step:", trainer.state.global_step)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch

Step,Training Loss
1,1.572240
2,1.574148
3,1.622022
4,1.698594


Final global_step: 4


Q10. Probability Assigned to Option E After Fine-Tuning
Using the fine-tuned LoRA model from Q9, run inference on row index 0 and apply softmax to the logits.

What is the probability assigned to Option E?

Round your answer to 4 decimal places.

In [14]:
# Set Model to Evaluation Mode
lora_model.eval()

# Run Inference
with torch.no_grad():
    outputs = lora_model(**enc0)


# Convert Logits to Probabilities

probabilities = torch.softmax(outputs.logits, dim=-1)[0]

print("Probabilities (A-E):", probabilities.tolist())
print("P(Option E):", round(probabilities[4].item(), 4))

Probabilities (A-E): [0.19969438016414642, 0.2001553624868393, 0.20062707364559174, 0.20007045567035675, 0.19945266842842102]
P(Option E): 0.1995
